In [2]:
# =============================================================================
# KAGGLE - 1-SLM COMPRESSED, FULL GSM8K TEST (1319), fp16
# =============================================================================
# NO RETRAINING. Your slm1_improved adapter is reused as-is.
#
# WHY REUSE IS VALID: a LoRA adapter is a pair of low-rank matrices whose shapes
# depend on layer in/out features, not on how the base was quantised. Training
# against a 4-bit base and serving on an fp16 base is standard QLoRA practice
# and usually helps slightly, because dequantisation noise is gone. Say so in
# the paper: "adapters trained with 4-bit QLoRA, evaluated in fp16".
#
# WHY fp16 MATTERS HERE: your earlier 68.0% was measured at 4-bit on n=150.
# Every number from today (71.6 / 72.1 / 45.5 / 56.0) is fp16 on n=1319. Mixing
# them injects a ~6-point precision artifact bigger than most of your effects.
# This run puts the 1-SLM on the same footing as everything else.
#
# TWO MEASUREMENTS, DELIBERATELY SEPARATE:
#   accuracy  - batched (fast), all 1319
#   latency   - sequential batch=1 on a 60-problem subset, because your 2-SLM
#               26.69s and Phase-B 16.05s were measured that way. Quoting a
#               batched latency against a sequential one would flatter the
#               1-SLM by ~5x and a reviewer would catch it.
#
# SETUP: add the slm1_improved folder as a Kaggle Dataset. Internet ON.
# =============================================================================

import os, re, csv, gc, json, math, time, glob, traceback
from collections import Counter

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
from datasets import load_dataset

import peft.import_utils
import peft.tuners.lora.torchao as _lora_torchao
peft.import_utils.is_torchao_available = lambda: False
_lora_torchao.is_torchao_available = lambda: False

# =============================================================================
# CONFIG
# =============================================================================
ADAPTER_DIR = None                      # auto-detected under /kaggle/input
BASE_MODEL  = "Qwen/Qwen2.5-Math-1.5B"  # fp16, same base the adapter was trained on
N_PROBLEMS  = 1319
BATCH_SIZE  = 16
MAX_NEW     = 128                        # compressed targets are ~5-19 words
LATENCY_N   = 60                         # sequential subset for the frontier number
OUT_CSV     = "/kaggle/working/slm1_compressed_full1319.csv"

# MUST match training exactly (CELL_5D). Ends "Solution:\n" - the trailing
# newline is the token-boundary fix; without it the model continues from a
# token it never saw.
PREFIX = ("Solve the problem. Show only the equations, then #### answer.\n"
          "Question: {question}\n"
          "Solution:\n")
assert PREFIX.endswith("Solution:\n")

# Reference points from today's fp16 runs, for the frontier table.
REF = [
    ("2-SLM guided (Qwen-Math)",      0.7160, 26.69),
    ("Phase B alone (Qwen-Math CoT)", 0.7207, 16.05),
    ("2-SLM guided (Qwen general)",   0.5600, None),
    ("Phase B alone (Qwen general)",  0.4550, None),
]


def find_adapter():
    if ADAPTER_DIR and os.path.exists(os.path.join(ADAPTER_DIR, "adapter_config.json")):
        return ADAPTER_DIR
    hits = glob.glob("/kaggle/input/**/adapter_config.json", recursive=True)
    # prefer a path that mentions the compressed model
    for h in hits:
        if "slm1" in h.lower() or "compress" in h.lower() or "improved" in h.lower():
            print(f"[find] adapter -> {os.path.dirname(h)}")
            return os.path.dirname(h)
    if hits:
        print(f"[find] adapter -> {os.path.dirname(hits[0])}  (only candidate)")
        return os.path.dirname(hits[0])
    raise FileNotFoundError("No adapter_config.json under /kaggle/input/. "
                            "Add slm1_improved as a Kaggle Dataset.")


ADAPTER = find_adapter()
with open(os.path.join(ADAPTER, "adapter_config.json")) as f:
    acfg = json.load(f)
print(f"[cfg] adapter base_model_name_or_path = {acfg.get('base_model_name_or_path')}")
print(f"[cfg] r={acfg.get('r')} alpha={acfg.get('lora_alpha')} "
      f"targets={acfg.get('target_modules')}")
mf = os.path.join(ADAPTER, "manifest.json")
if os.path.exists(mf):
    print(f"[cfg] training manifest: {json.load(open(mf))}")


# =============================================================================
# LOAD  (fp16 base + adapter, with vocab-mismatch guard)
# =============================================================================
# Training logged pad='<|PAD_TOKEN|>'(151665). If Unsloth ADDED that token it
# resized the embeddings, and the adapter then expects a larger vocab than a
# stock fp16 base has. Detect and resize rather than crash on a shape mismatch.
print("\n[load] fp16 base ...")
tok = AutoTokenizer.from_pretrained(ADAPTER, trust_remote_code=True)  # saved with training
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL, device_map="auto", low_cpu_mem_usage=True,
    trust_remote_code=True, torch_dtype=torch.float16)

emb = model.get_input_embeddings().weight.shape[0]
if len(tok) != emb:
    print(f"[vocab] tokenizer has {len(tok)} tokens, base embedding has {emb} "
          f"-> resizing to match the training-time vocabulary")
    model.resize_token_embeddings(len(tok))

model = PeftModel.from_pretrained(model, ADAPTER, is_trainable=False).eval()

if tok.pad_token_id is None:
    tok.pad_token = tok.eos_token
tok.padding_side = "left"          # required for batched decoder-only generation
if getattr(model, "generation_config", None) is not None:
    model.generation_config.max_length = None

EOS_IDS = sorted({i for i in [tok.eos_token_id,
                              tok.convert_tokens_to_ids("<|endoftext|>"),
                              tok.convert_tokens_to_ids("<|im_end|>")]
                  if isinstance(i, int) and i >= 0})
print(f"[load] eos={tok.eos_token!r} stops={EOS_IDS} pad={tok.pad_token!r}({tok.pad_token_id})")
print(f"[vram] {torch.cuda.memory_allocated()/1024**3:.2f} GB")


# =============================================================================
# PARSING  (matches CELL_5D: #### first, then last '=' result)
# =============================================================================
NUM_RE = re.compile(r"[-+]?\d[\d,]*(?:\.\d+)?")
HASH_RE = re.compile(r"####\s*([^\n]*)")
EQ_RE = re.compile(r"=\s*\$?\s*([-+]?\d[\d,]*(?:\.\d+)?)")
BOXED_RE = re.compile(r"\\boxed\s*\{([^{}]*)\}")


def to_f(t):
    if t is None:
        return None
    c = str(t).replace(",", "").replace("$", "").replace("\\", "").strip().rstrip(".")
    try:
        v = float(c)
    except ValueError:
        return None
    return v if math.isfinite(v) else None


def parse(raw):
    if not raw:
        return None, "empty"
    h = HASH_RE.findall(raw)
    if h:
        m = NUM_RE.search(h[-1])
        if m and to_f(m.group(0)) is not None:
            return to_f(m.group(0)), "hash"
    b = BOXED_RE.findall(raw)
    if b:
        m = NUM_RE.search(b[-1])
        if m and to_f(m.group(0)) is not None:
            return to_f(m.group(0)), "boxed"
    e = EQ_RE.findall(raw)
    if e and to_f(e[-1]) is not None:
        return to_f(e[-1]), "equals"
    n = NUM_RE.findall(raw)
    if n and to_f(n[-1]) is not None:
        return to_f(n[-1]), "last_number"
    return None, "none"


def gold_of(ans):
    m = NUM_RE.search(ans.split("####")[-1])
    return to_f(m.group(0)) if m else None


def ok(p, g):
    return p is not None and g is not None and math.isclose(p, g, rel_tol=1e-4, abs_tol=1e-4)


@torch.no_grad()
def gen(prompts, max_new=MAX_NEW):
    enc = tok(prompts, return_tensors="pt", padding=True,
              add_special_tokens=False).to(model.device)
    plen = enc["input_ids"].shape[1]
    out = model.generate(**enc, max_new_tokens=max_new, do_sample=False,
                         temperature=None, top_p=None, top_k=None,
                         eos_token_id=EOS_IDS, pad_token_id=tok.pad_token_id)
    return [t.strip() for t in tok.batch_decode(out[:, plen:], skip_special_tokens=True)]


def gen_safe(prompts, max_new=MAX_NEW):
    try:
        return gen(prompts, max_new)
    except torch.cuda.OutOfMemoryError:
        torch.cuda.empty_cache()
        if len(prompts) == 1:
            return [""]
        h = len(prompts) // 2
        return gen_safe(prompts[:h], max_new) + gen_safe(prompts[h:], max_new)


# =============================================================================
# ACCURACY - batched, all 1319
# =============================================================================
test = load_dataset("openai/gsm8k", "main", split="test")
N = min(N_PROBLEMS, len(test))
FIELDS = ["idx", "question", "gold", "raw", "pred", "strategy", "correct", "n_tokens"]

done = set()
if os.path.exists(OUT_CSV) and os.path.getsize(OUT_CSV) > 0:
    done = {int(r["idx"]) for r in csv.DictReader(open(OUT_CSV, newline=""))
            if r.get("idx", "").strip().isdigit()}
else:
    with open(OUT_CSV, "w", newline="", encoding="utf-8") as f:
        csv.DictWriter(f, fieldnames=FIELDS).writeheader()
todo = [i for i in range(N) if i not in done]
print(f"\n[run] {N} problems | {len(done)} done | {len(todo)} to go")

st, strat, toks = Counter(), Counter(), []
t0 = time.time()

for s in range(0, len(todo), BATCH_SIZE):
    ks = todo[s:s + BATCH_SIZE]
    qs = [" ".join(test[i]["question"].split()) for i in ks]
    gs = [gold_of(test[i]["answer"]) for i in ks]
    rows = []
    try:
        outs = gen_safe([PREFIX.replace("{question}", q) for q in qs])
        for i, q, g, raw in zip(ks, qs, gs, outs):
            p, sm = parse(raw)
            c = ok(p, g)
            nt = len(tok(raw, add_special_tokens=False)["input_ids"])
            toks.append(nt)
            st["n"] += 1
            st["ok"] += int(c)
            strat[sm] += 1
            rows.append({"idx": i, "question": q, "gold": g, "raw": raw,
                         "pred": p, "strategy": sm, "correct": c, "n_tokens": nt})
    except Exception as exc:
        traceback.print_exc(limit=1)
        torch.cuda.empty_cache()

    with open(OUT_CSV, "a", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=FIELDS, extrasaction="ignore")
        for r in rows:
            w.writerow({c2: r.get(c2, "") for c2 in FIELDS})
        f.flush()
        os.fsync(f.fileno())

    if st["n"]:
        el = time.time() - t0
        print(f"[{st['n']}/{len(todo)}] acc={st['ok']/st['n']:.1%} "
              f"{sum(toks)/len(toks):.0f}tok | ETA {(len(todo)-st['n'])*el/st['n']/60:.1f}m",
              flush=True)

ACC = st["ok"] / max(st["n"], 1)
print(f"\n[accuracy] {st['ok']}/{st['n']} = {ACC:.2%}")


# =============================================================================
# LATENCY - sequential batch=1, matching how 26.69s and 16.05s were measured
# =============================================================================
print(f"\n[latency] sequential batch=1 over {LATENCY_N} problems "
      f"(batched timing would not be comparable to the 2-SLM figures)")
lat = []
for i in range(min(LATENCY_N, N)):
    q = " ".join(test[i]["question"].split())
    t1 = time.time()
    gen_safe([PREFIX.replace("{question}", q)])
    lat.append(time.time() - t1)
LAT = sum(lat) / len(lat)
print(f"[latency] {LAT:.2f}s per problem (n={len(lat)})")


# =============================================================================
# SUMMARY + FRONTIER
# =============================================================================
n = max(st["n"], 1)
se = math.sqrt(ACC * (1 - ACC) / n)
print("\n" + "=" * 78)
print(f"1-SLM COMPRESSED (slm1_improved), fp16, n={st['n']}")
print(f"  accuracy   : {ACC:.2%}   95% CI [{ACC-1.96*se:.2%}, {ACC+1.96*se:.2%}]")
print(f"  latency    : {LAT:.2f}s/problem (sequential, batch=1)")
print(f"  tokens/ans : {sum(toks)/max(len(toks),1):.0f}")
print("  strategies :")
for k, v in strat.most_common():
    print(f"     {k:<16} {v:>5} ({v/n:.1%})")

print("\n" + "=" * 78)
print("ACCURACY / LATENCY FRONTIER  (all fp16, all n=1319 unless noted)")
print("=" * 78)
print(f"{'configuration':<34}{'acc':>8}{'latency':>11}{'speedup':>10}")
print("-" * 78)
base_lat = 26.69
for name, a, l in REF:
    ls = f"{l:.2f}s" if l else "n/a"
    sp = f"{base_lat/l:.1f}x" if l else "-"
    print(f"{name:<34}{a:>7.1%}{ls:>11}{sp:>10}")
print(f"{'1-SLM compressed (this run)':<34}{ACC:>7.1%}{LAT:>10.2f}s{base_lat/LAT:>9.1f}x")
print("-" * 78)
print(f"""
NOTE FOR THE PAPER
  Adapter trained with 4-bit QLoRA, evaluated in fp16 - state this. Your older
  68.0% figure was measured at 4-bit on n=150 and is NOT comparable to the
  numbers above; use this run instead.

  Latency here is sequential batch=1 to match how the 2-SLM 26.69s was measured.
  Do not quote the batched throughput from the accuracy phase alongside it.

  The claim to make is parity-or-better at ~{base_lat/LAT:.0f}x lower latency,
  with the CI, not a bare point estimate.

DOWNLOAD {OUT_CSV} BEFORE THE SESSION ENDS.
""")

[find] adapter -> /kaggle/input/datasets/tanishqborkar/slm1-improve/slm1_improved
[cfg] adapter base_model_name_or_path = unsloth/Qwen2.5-Math-1.5B-bnb-4bit
[cfg] r=32 alpha=64 targets=['q_proj', 'o_proj', 'up_proj', 'v_proj', 'k_proj', 'down_proj', 'gate_proj']
[cfg] training manifest: {'name': 'slm1_improved', 'final_train_loss': 0.13853098501314273, 'best_eval': 0.16990968585014343, 'minutes': 97.7, 'prefix': 'Solve the problem. Show only the equations, then #### answer.\nQuestion: {question}\nSolution:\n', 'lora_dropout': 0.05}

[load] fp16 base ...


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

[vocab] tokenizer has 151666 tokens, base embedding has 151936 -> resizing to match the training-time vocabulary


/usr/local/lib/python3.12/dist-packages/peft/config.py:220: UserWarning: Unexpected keyword arguments ['monteclora_config', 'velora_config'] for class LoraConfig, these are ignored. This probably means that you're loading a configuration file that was saved using a higher version of the library and additional parameters have been introduced since. It is highly recommended to upgrade the PEFT version before continuing (e.g. by running `pip install -U peft`).
  warnings.warn(


[load] eos='<|endoftext|>' stops=[151643, 151645] pad='<|PAD_TOKEN|>'(151665)
[vram] 3.30 GB


...

main/train-00000-of-00001.parquet:   0%|          | 0.00/2.31M [00:00<?, ?B/s]

main/test-00000-of-00001.parquet:   0%|          | 0.00/419k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/7473 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1319 [00:00<?, ? examples/s]


[run] 1319 problems | 0 done | 1319 to go
[16/1319] acc=62.5% 46tok | ETA 14.3m
[32/1319] acc=65.6% 41tok | ETA 12.5m
[48/1319] acc=66.7% 41tok | ETA 11.5m
[64/1319] acc=73.4% 40tok | ETA 11.7m
[80/1319] acc=77.5% 40tok | ETA 11.4m
[96/1319] acc=77.1% 39tok | ETA 11.5m
[112/1319] acc=74.1% 39tok | ETA 10.9m
[128/1319] acc=74.2% 38tok | ETA 10.8m
[144/1319] acc=74.3% 38tok | ETA 10.8m
[160/1319] acc=72.5% 39tok | ETA 10.6m
[176/1319] acc=71.6% 39tok | ETA 10.9m
[192/1319] acc=71.9% 40tok | ETA 11.0m
[208/1319] acc=72.1% 40tok | ETA 10.9m
[224/1319] acc=71.4% 40tok | ETA 10.7m
[240/1319] acc=72.5% 40tok | ETA 10.5m
[256/1319] acc=72.7% 40tok | ETA 10.3m
[272/1319] acc=73.2% 40tok | ETA 10.3m
[288/1319] acc=73.6% 40tok | ETA 10.0m
[304/1319] acc=73.0% 40tok | ETA 9.8m
[320/1319] acc=72.8% 40tok | ETA 9.6m
[336/1319] acc=72.0% 40tok | ETA 9.6m
[352/1319] acc=71.9% 40tok | ETA 9.3m
[368/1319] acc=71.2% 40tok | ETA 9.1m
[384/1319] acc=71.4% 40tok | ETA 8.9m
[400/1319] acc=71.5% 40tok | ETA 